## Leave-one-bank-out transfer

The FOMC arm asked whether 24 other central banks can stand in for scarce FOMC
labels. This repeats the design with the ECB and the Bank of England as the
held-out target. Each target is fine-tuned once on its own labels and once on
every other bank's, then scored on its own test split.

All text is lowercased, as in `wcb_transfer.ipynb`, since WCB is lowercase-only.
One seed, 78516, matching the FOMC arm.


### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

### Key Imports

In [ ]:
import torch

from config import RESULTS_DIR, SHAH_PLM
from data.loader_wcb_labelled import fetch_annotated
from models.plm_finetune import finetune
from sklearn.model_selection import train_test_split

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEED = 78516
TEST_FRAC = 0.2
TARGETS = {"ecb": "ecb", "boe": "bank_of_england", "boj": "bank_of_japan"}
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


### Split

The target bank is split 80/20, stratified by label. The borrowed arm trains on
every other bank in the corpus, which excludes the Fed since the loader drops it.


In [ ]:
wcb = fetch_annotated()
wcb = wcb.assign(sentence=wcb["sentence"].str.lower(), label=wcb["label_int"])
wcb = wcb[["bank_name", "sentence", "label"]]
print(f"wcb: {len(wcb):,} rows across {wcb['bank_name'].nunique()} banks")


def split(bank):
    own = wcb[wcb["bank_name"] == bank]
    train, test = train_test_split(
        own, test_size=TEST_FRAC, random_state=SEED, stratify=own["label"]
    )
    borrowed = wcb[wcb["bank_name"] != bank]
    return train, test, borrowed


for tag, bank in TARGETS.items():
    tr, te, bo = split(bank)
    print(f"{tag}: own train {len(tr)} | test {len(te)} | borrowed {len(bo):,}")
    print("   test balance", te["label"].value_counts().sort_index().to_dict())


### Fine-tune

Two arms per target, both scored on the same held-out test split, so only the
source of supervision varies.


In [ ]:
cfg = SHAH_PLM[ENC]

for tag, bank in TARGETS.items():
    train, test, borrowed = split(bank)
    for arm, train_df in [("own", train), ("borrowed", borrowed)]:
        model_key = f"{arm}:{ENC}"
        corpus = f"{tag}-lc"
        if already_done(OUT, force=FORCE, model=model_key, corpus=corpus, seed=SEED):
            print(f"{corpus} {model_key}: already done, skipping")
            continue
        print(f"{corpus} {model_key}: {len(train_df):,} training rows", flush=True)
        model, tok_, metrics = finetune(
            train_df,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=SEED,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus=corpus,
            seed=SEED,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{corpus} {model_key}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()


### Retention

Percentage of own-label performance that survives when the labels come from other
institutions. The FOMC row comes from `wcb_transfer.ipynb`, under corpus
`twd-lc`.


In [ ]:
import pandas as pd

r = pd.read_csv(OUT)
rows = [("fomc", "twd-lc", "roberta-large-lc", "wcb-only:roberta-large")]
rows += [(t, f"{t}-lc", "own:roberta-large", "borrowed:roberta-large") for t in TARGETS]

for tag, corpus, own_key, bor_key in rows:
    o = r[(r["corpus"] == corpus) & (r["model"] == own_key)]["macro_f1"]
    b = r[(r["corpus"] == corpus) & (r["model"] == bor_key)]["macro_f1"]
    if o.empty or b.empty:
        print(f"{tag:6s} pending")
        continue
    print(f"{tag:6s} own {o.mean():.4f}  borrowed {b.mean():.4f}  "
          f"retained {100 * b.mean() / o.mean():.0f}%")
